In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import stim
from itertools import combinations

In [ ]:
def build_golay24_H() -> np.ndarray:
    """Build H = [B | I_12] for the extended binary Golay code [24,12,8]."""
    B = np.zeros((12, 12), dtype=np.uint8)
    qr_11 = {1, 3, 4, 5, 9}

    # First row/col ones (except diagonal handled later)
    for i in range(1, 12):
        B[0, i] = 1
        B[i, 0] = 1

    # Quadratic residue rule for indices 1..11
    for i in range(1, 12):
        for j in range(1, 12):
            if i == j:
                B[i, j] = 1
            elif (j - i) % 11 in qr_11:
                B[i, j] = 1

    H = np.hstack([B, np.eye(12, dtype=np.uint8)]).astype(np.uint8)
    return H


def _pack_bits_to_int(bits: np.ndarray) -> int:
    """Pack a length-m array of 0/1 bits into an integer using little-endian order."""
    # bits[0] is LSB
    m = bits.size
    out = 0
    for i in range(m):
        out |= (int(bits[i]) & 1) << i
    return out


def build_syndrome_correction_table_golay24(H: np.ndarray) -> np.ndarray:
    """
    Build a complete correction lookup table for extended Golay [24,12,8].

    Returns:
      corr_table: np.uint32 array of shape (4096,)
        corr_table[s] is a 24-bit mask to XOR with the measured data bits.

    Strategy:
      Fill syndromes with minimal-weight coset leaders by enumerating error masks
      in increasing weight order up to weight 4 (covering radius 4).
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape  # m=12, n=24
    assert m == 12 and n == 24

    # Pack each column of H into a 12-bit integer
    col_int = np.zeros(n, dtype=np.uint16)
    for j in range(n):
        col_int[j] = _pack_bits_to_int(H[:, j])

    # Correction table indexed by 12-bit syndrome (0..4095)
    corr_table = np.zeros(1 << m, dtype=np.uint32)
    filled = np.zeros(1 << m, dtype=bool)

    # Weight 0
    filled[0] = True
    corr_table[0] = 0

    # Helper to set entry if empty
    def try_set(syn: int, mask: int):
        if not filled[syn]:
            filled[syn] = True
            corr_table[syn] = np.uint32(mask)

    # Weight 1
    for i in range(n):
        syn = int(col_int[i])
        mask = 1 << i
        try_set(syn, mask)

    # Weight 2
    for i, j in combinations(range(n), 2):
        syn = int(col_int[i] ^ col_int[j])
        mask = (1 << i) ^ (1 << j)
        try_set(syn, mask)

    # Weight 3
    for i, j, k in combinations(range(n), 3):
        syn = int(col_int[i] ^ col_int[j] ^ col_int[k])
        mask = (1 << i) ^ (1 << j) ^ (1 << k)
        try_set(syn, mask)

    # Weight 4 (covering radius for extended Golay)
    for i, j, k, l in combinations(range(n), 4):
        syn = int(col_int[i] ^ col_int[j] ^ col_int[k] ^ col_int[l])
        mask = (1 << i) ^ (1 << j) ^ (1 << k) ^ (1 << l)
        try_set(syn, mask)

    if not np.all(filled):
        missing = np.where(~filled)[0]
        raise RuntimeError(f"Correction table incomplete, missing {missing.size} syndromes.")

    return corr_table


H = build_golay24_H()
corr_table = build_syndrome_correction_table_golay24(H)

print("H shape:", H.shape)
print("HH^T mod 2 == 0 ?", bool(np.all((H @ H.T) % 2 == 0)))
print("Correction table size:", corr_table.size, "(expected 4096)")

In [ ]:
def golay24_x_memory_circuit(H: np.ndarray, p: float) -> stim.Circuit:
    """
    One-shot X-error memory with Z-parity checks defined by H.

    Qubits:
      data: 0..23
      anc:  24..35  (12 ancillas, one per check row)

    Noise:
      X_ERROR(p) on each data qubit (independent bit-flips).

    Measurements (in order):
      12 ancilla Z measurements (syndrome bits)
      24 data Z measurements (data bits after noise, before correction)
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert m == 12 and n == 24

    data = list(range(n))
    anc = list(range(n, n + m))  # 24..35

    c = stim.Circuit()

    # Prepare |0...0> for data and ancillas
    c.append("R", data + anc)

    # Apply bit-flip noise to data qubits
    c.append("X_ERROR", data, p)

    # Parity checks: ancilla accumulates XOR of selected data bits
    # Implemented with CX from data -> anc
    for row in range(m):
        ops = []
        a = anc[row]
        ones = np.where(H[row] == 1)[0]
        for q in ones:
            ops += [int(q), int(a)]
        if ops:
            c.append("CX", ops)

    # Measure ancillas (syndrome)
    c.append("M", anc)

    # Measure data (so decoding success can be evaluated)
    c.append("M", data)

    return c

In [ ]:
def simulate_golay24_quantum(H: np.ndarray, corr_table: np.ndarray, p: float, shots: int, seed: int = 0) -> float:
    """
    Quantum simulation using Stim:
      - sample syndrome bits from ancilla measurements
      - sample data bits from data measurements
      - decode with corr_table (syndrome -> correction mask)
      - count block failure if residual != 0

    Returns:
      block_error_rate
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert m == 12 and n == 24
    assert corr_table.size == 4096

    c = golay24_x_memory_circuit(H, p)
    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots)  # shape: (shots, 12 + 24)

    synd = ms[:, :m].astype(np.uint16)
    data = ms[:, m:].astype(np.uint32)

    # Vectorized packing of bits into ints (little-endian)
    w12 = (1 << np.arange(m, dtype=np.uint16))
    w24 = (1 << np.arange(n, dtype=np.uint32))

    syn_int = (synd @ w12).astype(np.uint16)     # (shots,)
    data_int = (data @ w24).astype(np.uint32)    # (shots,)

    corr_int = corr_table[syn_int]               # (shots,)
    residual = data_int ^ corr_int

    ber = float(np.mean(residual != 0))
    return ber


# --- Parameters ---
p_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.08, 0.10, 0.12, 0.15]
shots = 1_000_000

# --- Print an example circuit (p=0.01) ---
c_demo = golay24_x_memory_circuit(H, p=0.01)
print(c_demo)

# --- Run simulation ---
bers = []
for p in p_values:
    ber = simulate_golay24_quantum(H, corr_table, p, shots, seed=12345)
    bers.append(ber)
    print(f"p={p:.3f} -> block error rate = {ber:.6e}")

# --- Theoretical curve for extended Golay [24,12,8] under BSC(p) with guaranteed unique decoding up to t=3 ---
# Failure occurs when >= 4 bit flips happen (because d=8 -> t=floor((d-1)/2)=3)
import math

theory = []
for p in p_values:
    ok = 0.0
    for k in range(0, 4):
        ok += math.comb(24, k) * (p**k) * ((1 - p)**(24 - k))
    theory.append(1.0 - ok)

# --- Plot ---
plt.figure(figsize=(8, 5))
plt.semilogy(p_values, [max(x, 1e-12) for x in bers], 'o-', label='Sim (Stim) + syndrome decoding')
plt.semilogy(p_values, [max(x, 1e-12) for x in theory], 's-', label='Theory Golay [24,12,8] (t=3): P(W ≥ 4)')
plt.semilogy(p_values, p_values, '--', label='Physical p (no correction)')
plt.xlabel('Physical bit-flip probability p')
plt.ylabel('Block error rate')
plt.title('Extended Golay [24,12,8] – quantum (Stim) vs theoretical bound')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

p_demo = 0.01
circuito_demo = golay24_x_memory_circuit(H, p_demo)

circuito_demo.diagram('timeline-svg')



In [ ]:
# Golay(24,12,8): p_L vs p using hierarchical Monte Carlo concatenation (L=1,2,3)
# L=1 uses the Stim circuit.
# L=2 and L=3 are built by sampling 24 outputs from the previous level and decoding with the outer Golay decoder.

import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Configuration
# -------------------------
levels = [1, 2, 3]
eps = 1e-12

# Physical error rates (log-spaced, where Golay helps)
p_min, p_max = 1e-4, 8e-2
num_points = 24
p_values = np.geomspace(p_min, p_max, num_points)

# Shots per level (keep L2/L3 smaller; L1 is the expensive Stim part)
shots_L1 = 300_000
shots_L2 = 200_000
shots_L3 = 100_000

seed_base = 12345

# -------------------------
# Helpers
# -------------------------
def sample_level1_fail_bits_from_stim(H: np.ndarray, corr_table: np.ndarray, p: float, shots: int, seed: int) -> np.ndarray:
    """
    Run the Stim circuit for Golay(24,12,8) X-memory and return a 0/1 array:
      fail=1 if decoding does NOT return the all-zero codeword (i.e., corrected data != 0).
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert (m, n) == (12, 24)
    assert corr_table.size == 4096

    c = golay24_x_memory_circuit(H, p)
    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots)  # shape: (shots, 12 + 24), bits are 0/1

    synd = ms[:, :m].astype(np.uint16)
    data = ms[:, m:].astype(np.uint32)

    # Pack bits into ints (little-endian)
    w12 = (1 << np.arange(m, dtype=np.uint16))
    w24 = (1 << np.arange(n, dtype=np.uint32))

    syn_int = (synd @ w12).astype(np.uint16)      # (shots,)
    data_int = (data @ w24).astype(np.uint32)     # (shots,)

    corr_int = corr_table[syn_int].astype(np.uint32)
    corrected = data_int ^ corr_int               # should be 0 if fully corrected (all-zero codeword)

    fail_bits = (corrected != 0).astype(np.uint8)
    return fail_bits


def decode_outer_fail_bits(H: np.ndarray, corr_table: np.ndarray, err_bits: np.ndarray) -> np.ndarray:
    """
    Outer Golay decoder: given err_bits of shape (shots, 24) (0/1),
    compute syndrome, apply correction, and return fail bits:
      fail=1 if residual != 0 after decoding.
    """
    H = (H.copy() & 1).astype(np.uint8)
    m, n = H.shape
    assert (m, n) == (12, 24)
    assert err_bits.shape[1] == 24
    assert corr_table.size == 4096

    err_bits = err_bits.astype(np.uint8)

    # Syndrome bits and packing
    synd = (err_bits @ H.T) % 2                   # (shots, 12)
    w12 = (1 << np.arange(m, dtype=np.uint16))
    syn_int = (synd.astype(np.uint16) @ w12).astype(np.uint16)

    # Pack error bits into int
    w24 = (1 << np.arange(n, dtype=np.uint32))
    err_int = (err_bits.astype(np.uint32) @ w24).astype(np.uint32)

    corr_int = corr_table[syn_int].astype(np.uint32)
    residual = err_int ^ corr_int

    fail_bits = (residual != 0).astype(np.uint8)
    return fail_bits


# -------------------------
# Main loop: compute p_L for L=1,2,3
# -------------------------
curves = {L: [] for L in levels}

rng = np.random.default_rng(seed_base)

for idx, p in enumerate(p_values):
    # ---- Level 1: true quantum simulation (Stim circuit) ----
    fail1 = sample_level1_fail_bits_from_stim(H, corr_table, float(p), shots_L1, seed=seed_base + 1000 + idx)
    pL1 = float(fail1.mean())
    curves[1].append(pL1)

    # ---- Level 2: sample 24 outputs from level 1 and decode outer ----
    if 2 in levels:
        # Bootstrap: build (shots_L2, 24) by sampling from level-1 outcomes
        err2 = rng.choice(fail1, size=(shots_L2, 24), replace=True)
        fail2 = decode_outer_fail_bits(H, corr_table, err2)
        pL2 = float(fail2.mean())
        curves[2].append(pL2)

    # ---- Level 3: sample 24 outputs from level 2 and decode outer ----
    if 3 in levels:
        err3 = rng.choice(fail2, size=(shots_L3, 24), replace=True)
        fail3 = decode_outer_fail_bits(H, corr_table, err3)
        pL3 = float(fail3.mean())
        curves[3].append(pL3)

    print(f"p={p:.3e} -> L1={pL1:.3e}" +
          (f", L2={pL2:.3e}" if 2 in levels else "") +
          (f", L3={pL3:.3e}" if 3 in levels else ""))

# Convert lists to arrays
for L in levels:
    curves[L] = np.array(curves[L], dtype=float)

# -------------------------
# Plot (log-log)
# -------------------------
plt.figure(figsize=(9, 5))

for L in levels:
    n_phys = 24 ** L
    k_log  = 12 ** L
    label = f"Golay hierarchical L={L} (n={n_phys}, k={k_log}, k/n={k_log/n_phys:.3f})"
    plt.loglog(p_values, np.maximum(curves[L], eps), "o-", label=label)

plt.loglog(p_values, p_values, "--", label="No coding: $p_L = p$")
plt.xlabel("Physical error rate p")
plt.ylabel("Logical / block failure rate $p_L$")
plt.title("Golay(24,12,8): $p_L$ vs $p$ (Stim L1 + hierarchical outer decoding)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# True L=2 (n=24^2) construction: Golay(24,12,8) product code on a 24x24 grid (576 data qubits).
# Quantum part: Stim circuit generates X errors via X_ERROR(p) and measures all 576 data qubits.
# Classical part: iterative row/column decoding using the Golay syndrome table (corr_table).

import numpy as np
import matplotlib.pyplot as plt
import stim

# ----------------------------
# Parameters (tune as needed)
# ----------------------------
p_values = np.geomspace(3e-3, 2e-1, 18)
shots_L1 = 200_000
shots_L2 = 200_000
num_iters = 2
batch_size = 5_000
num_iters = 2           # number of row/col decoding sweeps
seed_base = 12345
eps = 1e-12

# ----------------------------
# Sanity checks on inputs
# ----------------------------
H = (H.copy() & 1).astype(np.uint8)
assert H.shape == (12, 24)
assert corr_table.size == 4096

# Precompute correction bits for fast XOR:
# corr_bits[s] gives a length-24 bit-vector correction for syndrome s.
corr_bits = ((corr_table[:, None] >> np.arange(24, dtype=np.uint32)) & 1).astype(np.uint8)  # (4096, 24)
w12 = (1 << np.arange(12, dtype=np.uint16))  # weights for packing syndrome bits into an int

def golay_product_L2_circuit(p: float) -> stim.Circuit:
    """
    24x24 data qubits: indices 0..575 mapped as q(r,c)=r*24+c.
    Initialize to |0>, apply X_ERROR(p) to all data qubits, measure all.
    """
    n = 24 * 24
    c = stim.Circuit()
    data = list(range(n))
    c.append("R", data)
    c.append("X_ERROR", data, p)
    c.append("M", data)
    return c

def _decode_rows_inplace(grid: np.ndarray) -> None:
    """
    Decode each of the 24 rows as an independent Golay(24,12,8) block.
    grid shape: (B, 24, 24), uint8 bits.
    """
    B = grid.shape[0]
    rows = grid.reshape(B * 24, 24)                      # (B*24, 24)
    synd = (rows @ H.T) & 1                              # (B*24, 12) over GF(2)
    syn_int = (synd.astype(np.uint16) @ w12).astype(np.uint16)   # (B*24,)
    rows ^= corr_bits[syn_int]                           # apply correction
    grid[:] = rows.reshape(B, 24, 24)

def _decode_cols_inplace(grid: np.ndarray) -> None:
    """
    Decode each of the 24 columns as an independent Golay(24,12,8) block.
    grid shape: (B, 24, 24), uint8 bits.
    """
    B = grid.shape[0]
    gT = grid.transpose(0, 2, 1)                         # (B, 24, 24) now "rows" are original columns
    cols = gT.reshape(B * 24, 24)                        # (B*24, 24)
    synd = (cols @ H.T) & 1                              # (B*24, 12)
    syn_int = (synd.astype(np.uint16) @ w12).astype(np.uint16)
    cols ^= corr_bits[syn_int]
    grid[:] = cols.reshape(B, 24, 24).transpose(0, 2, 1)

def simulate_golay_product_L2_quantum(p: float, shots: int, seed: int = 0, iters: int = 2) -> float:
    """
    Returns block failure rate for the 24x24 Golay×Golay product code.
    Failure if residual grid != 0 after iterative row/column decoding.
    """
    c = golay_product_L2_circuit(p)
    sampler = c.compile_sampler(seed=seed)

    fails = 0
    done = 0

    while done < shots:
        B = min(batch_size, shots - done)
        ms = sampler.sample(B).astype(np.uint8)          # (B, 576) bits
        grid = ms.reshape(B, 24, 24)                     # map into 24x24

        # Iterative product-code decoding: rows then cols (repeat)
        for _ in range(iters):
            _decode_rows_inplace(grid)
            _decode_cols_inplace(grid)

        # Success if all-zero after decoding
        batch_fails = np.any(grid.reshape(B, -1), axis=1).sum()
        fails += int(batch_fails)
        done += B

    return fails / shots

# ----------------------------
# Run: L=1 (optional) + true L=2
# ----------------------------
pL1 = []
pL2 = []

for i, p in enumerate(p_values):
    # L=1 from your existing function (Stim on 24 qubits + Golay decoding)
    ber1 = simulate_golay24_quantum(H, corr_table, float(p), shots_L1, seed=seed_base + 10_000 + i)
    pL1.append(ber1)

    # True L=2 product code on 576 qubits
    ber2 = simulate_golay_product_L2_quantum(float(p), shots_L2, seed=seed_base + 20_000 + i, iters=num_iters)
    pL2.append(ber2)

    print(f"p={p:.3e} -> L1={ber1:.3e},  L2(true)={ber2:.3e}")

pL1 = np.array(pL1, dtype=float)
pL2 = np.array(pL2, dtype=float)

# ----------------------------
# Plot (log-log)
# ----------------------------
plt.figure(figsize=(9, 5))
plt.loglog(p_values, np.maximum(pL1, eps), "o-", label="L=1 Golay(24,12,8) (Stim)")
plt.loglog(p_values, np.maximum(pL2, eps), "s-", label=f"L=2 TRUE (Golay×Golay product, iters={num_iters})")
plt.loglog(p_values, p_values, "--", label="No coding: $p_L=p$")
plt.xlabel("Physical error rate p")
plt.ylabel("Block failure rate $p_L$")
plt.title("Golay: L=1 vs TRUE L=2 (24×24 product code) under X_ERROR(p)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import stim
from IPython.display import display
from itertools import combinations

# ============================================================
# Golay(24,12,8) X-only "memory" demo (NO pre-EC, NO plots):
# - Scenario A: only injected X on one data qubit, no X_ERROR(p)
# - Scenario B: only stochastic X_ERROR(p), no injected X
# Prints: circuit diagrams + error rates for both scenarios
# ============================================================

# ----------------------------
# Build extended Golay parity-check H = [B | I] (12x24)
# ----------------------------
def build_golay24_H() -> np.ndarray:
    """Build H = [B | I_12] for the extended binary Golay code [24,12,8]."""
    B = np.zeros((12, 12), dtype=np.uint8)
    qr_11 = {1, 3, 4, 5, 9}

    for i in range(1, 12):
        B[0, i] = 1
        B[i, 0] = 1

    for i in range(1, 12):
        for j in range(1, 12):
            if i == j:
                B[i, j] = 1
            elif (j - i) % 11 in qr_11:
                B[i, j] = 1

    return np.hstack([B, np.eye(12, dtype=np.uint8)]).astype(np.uint8)

def _pack_bits_to_int(bits: np.ndarray) -> int:
    """Pack a length-m array of 0/1 bits into an integer using little-endian order."""
    out = 0
    for i, b in enumerate(bits.tolist()):
        out |= (int(b) & 1) << i
    return out

def build_corr_table_golay24(Hm: np.ndarray) -> np.ndarray:
    """
    Complete syndrome->correction table for extended Golay by enumerating coset leaders
    in increasing weight up to 4 (covering radius 4), so all 2^12 syndromes are covered.

    Returns:
      corr_table[s] = 24-bit mask (uint32) to XOR with measured data bits.
    """
    Hm = (Hm.copy() & 1).astype(np.uint8)
    m, n = Hm.shape
    assert (m, n) == (12, 24)

    # Pack columns of H as 12-bit integers
    col_int = np.zeros(n, dtype=np.uint16)
    for j in range(n):
        col_int[j] = _pack_bits_to_int(Hm[:, j])

    corr_table = np.zeros(1 << m, dtype=np.uint32)
    filled = np.zeros(1 << m, dtype=bool)

    filled[0] = True
    corr_table[0] = 0

    def try_set(syn: int, mask: int):
        if not filled[syn]:
            filled[syn] = True
            corr_table[syn] = np.uint32(mask)

    # Weight 1
    for i in range(n):
        try_set(int(col_int[i]), 1 << i)

    # Weight 2
    for i, j in combinations(range(n), 2):
        try_set(int(col_int[i] ^ col_int[j]), (1 << i) ^ (1 << j))

    # Weight 3
    for i, j, k in combinations(range(n), 3):
        try_set(int(col_int[i] ^ col_int[j] ^ col_int[k]), (1 << i) ^ (1 << j) ^ (1 << k))

    # Weight 4
    for i, j, k, l in combinations(range(n), 4):
        try_set(int(col_int[i] ^ col_int[j] ^ col_int[k] ^ col_int[l]),
                (1 << i) ^ (1 << j) ^ (1 << k) ^ (1 << l))

    if not np.all(filled):
        missing = np.where(~filled)[0]
        raise RuntimeError(f"Correction table incomplete, missing {missing.size} syndromes.")

    return corr_table

def append_z_syndrome_extraction(c: stim.Circuit, Hm: np.ndarray, data: list[int], anc: list[int]):
    """
    Measure Z-parity checks defined by H using ancillas in |0> and CX(data->anc).
    Produces 12 syndrome bits in order.
    """
    c.append("R", anc)
    for row in range(12):
        a = anc[row]
        ones = np.where(Hm[row] == 1)[0]
        if ones.size:
            ops = []
            for q in ones:
                ops += [data[int(q)], a]
            c.append("CX", ops)
    c.append("M", anc)

# ----------------------------
# Circuit builder (NO pre-EC)
# ----------------------------
def build_golay24_demo_circuit(
    p_noise: float = 0.0,
    inject_error: bool = False,
    injected_qubit: int = 2,
) -> stim.Circuit:
    """
    Single Golay block:
      data: 0..23
      anc:  24..35

    Flow:
      - reset
      - (start from |0..0> = zero codeword)
      - optionally inject X on data[injected_qubit]
      - optionally apply X_ERROR(p_noise) on all 24 data qubits
      - syndrome extraction (12 checks)
      - measure data (24 bits)
    """
    Hm = build_golay24_H()
    data = list(range(24))
    anc  = list(range(24, 36))

    c = stim.Circuit()
    c.append("R", data + anc)

    if inject_error:
        if not (0 <= injected_qubit <= 23):
            raise ValueError("injected_qubit must be in [0..23].")
        c.append("X", [injected_qubit])

    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    append_z_syndrome_extraction(c, Hm, data, anc)
    c.append("M", data)

    return c

def run_and_score_golay24(
    c: stim.Circuit,
    corr_table: np.ndarray,
    shots: int,
    seed: int = 12345
) -> float:
    """
    Runs the circuit, decodes using corr_table, returns block failure rate:
      fail = 1 if (measured_data XOR correction(syndrome)) != 0
    """
    sampler = c.compile_sampler(seed=seed)
    ms = sampler.sample(shots).astype(np.uint8)

    # Layout per shot: syndrome (12) + data (24)
    synd = ms[:, 0:12].astype(np.uint16)
    data = ms[:, 12:36].astype(np.uint32)

    # Pack syndrome bits to int (little-endian)
    w12 = (1 << np.arange(12, dtype=np.uint16))
    syn_int = (synd @ w12).astype(np.uint16)

    # Pack data bits to int (little-endian)
    w24 = (1 << np.arange(24, dtype=np.uint32))
    data_int = (data @ w24).astype(np.uint32)

    corr_int = corr_table[syn_int].astype(np.uint32)
    corrected = data_int ^ corr_int

    return float(np.mean(corrected != 0))

# ----------------------------
# SETTINGS
# ----------------------------
shots = 100_000
injected_qubit = 2
p_noise = 0.01

Hm = build_golay24_H()
corr_table = build_corr_table_golay24(Hm)

# ============================================================
# Scenario A: injected-only (single X), no X_ERROR(p)
# ============================================================
c_injected_only = build_golay24_demo_circuit(
    p_noise=0.0,
    inject_error=True,
    injected_qubit=injected_qubit,
)

print("=== Scenario A (Golay24): injected-only (single X), no X_ERROR(p) ===")
display(c_injected_only.diagram("timeline-svg"))
rate_A = run_and_score_golay24(c_injected_only, corr_table, shots=shots, seed=111)
print(f"injected_qubit={injected_qubit}, shots={shots}")
print(f"Block failure rate = {rate_A:.6e}\n")

# ============================================================
# Scenario B: noise-only (X_ERROR(p)), no injected error
# ============================================================
c_noise_only = build_golay24_demo_circuit(
    p_noise=p_noise,
    inject_error=False,
)

print("=== Scenario B (Golay24): noise-only (X_ERROR(p)), no injected X ===")
display(c_noise_only.diagram("timeline-svg"))
rate_B = run_and_score_golay24(c_noise_only, corr_table, shots=shots, seed=222)
print(f"p_noise={p_noise}, shots={shots}")
print(f"Block failure rate = {rate_B:.6e}")

In [ ]:
import numpy as np
import stim
from IPython.display import display

# ============================================================
# Extended Golay (24,12,8) "H bias trap" demo (Clifford-only, Stim)
#
# Key idea:
#   - Measuring X-checks on |0^24> has an unknown baseline.
#   - So we record a reference X-syndrome (prepX) and later use syndrome DIFFERENCES.
#
# We compare:
#   (A) Control:   prepX -> X_ERROR(p) -> measure postZ, postX
#   (B) With H:    prepX -> X_ERROR(p) -> H^{⊗24} -> measure postZ, postX
#
# Interpretation:
#   - Z-check outcomes (postZ) detect X-errors (bit flips).
#   - X-check outcomes (postX) detect Z-errors (phase flips), but must be baseline-subtracted.
#   - Under transversal H: X <-> Z, so:
#       * Z-errors after H show up directly in postX (baseline becomes 0).
#       * X-errors after H show up in postZ, but baseline is prepX (so XOR it).
# ============================================================

def build_golay24_H() -> np.ndarray:
    """Build H = [B | I_12] for the extended binary Golay code [24,12,8]."""
    B = np.zeros((12, 12), dtype=np.uint8)
    qr_11 = {1, 3, 4, 5, 9}

    # First row/col ones
    for i in range(1, 12):
        B[0, i] = 1
        B[i, 0] = 1

    # Quadratic residue rule on indices 1..11
    for i in range(1, 12):
        for j in range(1, 12):
            if i == j:
                B[i, j] = 1
            elif (j - i) % 11 in qr_11:
                B[i, j] = 1

    return np.hstack([B, np.eye(12, dtype=np.uint8)]).astype(np.uint8)

def append_Z_stabilizers_golay(c: stim.Circuit, Hm: np.ndarray, data: list[int], ancZ: list[int]):
    """Measure Z-type stabilizers (detect X component) using |0> ancillas and M."""
    c.append("R", ancZ)
    for row in range(12):
        a = ancZ[row]
        ones = np.where(Hm[row] == 1)[0]
        if ones.size:
            ops = []
            for q in ones:
                ops += [data[int(q)], a]  # CX data -> anc
            c.append("CX", ops)
    c.append("M", ancZ)

def append_X_stabilizers_golay(c: stim.Circuit, Hm: np.ndarray, data: list[int], ancX: list[int]):
    """Measure X-type stabilizers (detect Z component) using |+> ancillas and MX."""
    c.append("R", ancX)
    c.append("H", ancX)  # prepare |+>
    for row in range(12):
        a = ancX[row]
        ones = np.where(Hm[row] == 1)[0]
        if ones.size:
            ops = []
            for q in ones:
                ops += [a, data[int(q)]]  # CX anc -> data
            c.append("CX", ops)
    c.append("MX", ancX)

def build_golay_H_bias_circuit(p_noise: float, apply_H: bool) -> stim.Circuit:
    Hm = build_golay24_H()
    data = list(range(0, 24))

    # Separate ancillas so measurement slices are clean
    ancX0 = list(range(24, 36))   # prepX (12)
    ancZ  = list(range(36, 48))   # postZ (12)
    ancX1 = list(range(48, 60))   # postX (12)

    c = stim.Circuit()
    c.append("R", data + ancX0 + ancZ + ancX1)

    # Reference X-syndrome (baseline)
    append_X_stabilizers_golay(c, Hm, data, ancX0)  # 12 bits: prepX

    # X-only noise before the gate
    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    # Optional transversal H
    if apply_H:
        c.append("H", data)

    # Post syndromes
    append_Z_stabilizers_golay(c, Hm, data, ancZ)   # 12 bits: postZ
    append_X_stabilizers_golay(c, Hm, data, ancX1)  # 12 bits: postX

    return c

def run_golay_H_bias_demo(p_noise: float, shots: int, seed: int = 12345, show_diagrams: bool = True):
    Hm = build_golay24_H()
    assert Hm.shape == (12, 24)

    def analyze(c: stim.Circuit, label: str, apply_H: bool):
        sampler = c.compile_sampler(seed=seed)
        ms = sampler.sample(shots).astype(np.uint8)

        # Layout: prepX(12) | postZ(12) | postX(12)  => total 36
        prepX = ms[:, 0:12]
        postZ = ms[:, 12:24]
        postX = ms[:, 24:36]

        if apply_H:
            # After H: X<->Z swap.
            # - Z-errors show up directly in postX (baseline becomes 0).
            # - X-errors show up in postZ but baseline is prepX.
            x_synd = postZ ^ prepX
            z_synd = postX
        else:
            # No H: Z-stabilizers baseline is 0; X-stabilizers baseline is prepX.
            x_synd = postZ
            z_synd = postX ^ prepX

        trig_x = np.mean(np.any(x_synd != 0, axis=1)) * 100
        trig_z = np.mean(np.any(z_synd != 0, axis=1)) * 100

        print(f"\n=== {label} ===")
        print(f"p_noise = {p_noise:.4f} | shots = {shots}")
        print(f"X-syndrome triggers (detect X component): {trig_x:.2f}%")
        print(f"Z-syndrome triggers (detect Z component): {trig_z:.2f}%")

        return trig_x, trig_z

    # Control (no H)
    c0 = build_golay_H_bias_circuit(p_noise=p_noise, apply_H=False)
    # With H
    cH = build_golay_H_bias_circuit(p_noise=p_noise, apply_H=True)

    if show_diagrams:
        print("\nCircuit: CONTROL (no H)")
        display(c0.diagram("timeline-svg"))
        print("\nCircuit: WITH transversal H")
        display(cH.diagram("timeline-svg"))

    analyze(c0, "CONTROL: no H", apply_H=False)
    analyze(cH, "WITH H: transversal H (X -> Z)", apply_H=True)

# --- RUN ---
p_noise = 0.05
shots = 20_000
run_golay_H_bias_demo(p_noise=p_noise, shots=shots, seed=12345, show_diagrams=True)

### The Hadamard Trap at Scale: Validating the Golay `[24,12,8]` Architecture

Having established the theoretical rules on the Steane code, we must validate these constraints on our actual target architecture: the Extended Golay `[24,12,8]` code. Scaling up to 24 physical qubits introduces new complexities, both in accumulated error probabilities and in state preparation.

**1. Methodology: Syndrome Differencing**
In this simulation, we introduced a highly efficient, hardware-friendly tracking technique. Preparing a pristine $|0_L\rangle$ state for a 24-qubit code via projection and post-selection would discard an overwhelming majority of experimental shots. Instead, we use **Syndrome Differencing**: we initialize the data in $|0^{24}\rangle$, record its baseline X-syndrome ($prepX$), apply the noise and gates, and then XOR the final syndrome with the baseline. This effectively subtracts the initial state's non-zero parity baseline, perfectly isolating the errors injected during the cycle without dropping a single shot.



**2. The Baseline: High-Density Noise (Control Scenario)**
We injected a realistic Cat Qubit pure bit-flip ($X$) noise rate of $p=5\%$. 
* **Result Interpretation:** At $p=0.05$ across 24 qubits, the statistical probability of at least one physical error occurring in the block is $1 - (1 - 0.05)^{24} \approx 70.8\%$. Our output perfectly matches this: the $X$-syndrome detectors triggered at **71.03%**. This proves our Golay extraction circuit is actively and accurately "seeing" the physical $X$ errors, while the $Z$-error detectors remain safely at **0.00%**.

**3. The Conjugation Disaster at Scale (With H Scenario)**
We then applied a direct transversal Hadamard gate across the 24 noisy qubits.
* **Result Interpretation:** The mathematical reality of $H \cdot X \cdot H = Z$ scales mercilessly. The $X$-syndrome triggers instantly plummeted to 0.00%, and the $Z$-syndrome triggers spiked to **71.03%**. 



**Conclusion**
This 24-qubit simulation provides the definitive architectural verdict:
Applying transversal axis-mixing gates directly to a large Cat Qubit memory block will instantaneously flood the system with uncorrectable phase errors. Because the Golay `[24,12,8]` code is designed to correct up to 3 errors, hitting it with a 71% chance of block-wide $Z$-errors ensures immediate logical failure. 

Therefore, our **Gate Teleportation Protocol** (via transversal CNOTs) is not just a theoretical optimization; it is the absolute prerequisite for scaling Cat Qubits into a universal fault-tolerant quantum computer.

In [ ]:
import numpy as np
import stim
from IPython.display import display

# ============================================================
# Extended Golay (24,12,8) "S bias trap" demo (Clifford-only, Stim)
#
# We compare:
#   (A) Control: prepX -> X_ERROR(p) -> measure postZ, postX
#   (B) With S:  prepX -> X_ERROR(p) -> S^{⊗24} -> measure postZ, postX
#
# Interpretation:
#   - postZ detects X component directly (baseline 0).
#   - postX detects Z component, but baseline-subtracted via XOR with prepX.
#   - Under transversal S: X -> Y, so a pure X error becomes X+Z component:
#       => both syndromes trigger, and (often) syndromes match shot-by-shot.
# ============================================================

def build_golay24_H() -> np.ndarray:
    """Build H = [B | I_12] for the extended binary Golay code [24,12,8]."""
    B = np.zeros((12, 12), dtype=np.uint8)
    qr_11 = {1, 3, 4, 5, 9}

    for i in range(1, 12):
        B[0, i] = 1
        B[i, 0] = 1

    for i in range(1, 12):
        for j in range(1, 12):
            if i == j:
                B[i, j] = 1
            elif (j - i) % 11 in qr_11:
                B[i, j] = 1

    return np.hstack([B, np.eye(12, dtype=np.uint8)]).astype(np.uint8)

def append_Z_stabilizers_golay(c: stim.Circuit, Hm: np.ndarray, data: list[int], ancZ: list[int]):
    """Measure Z-type stabilizers (detect X component) using |0> ancillas and M."""
    c.append("R", ancZ)
    for row in range(12):
        a = ancZ[row]
        ones = np.where(Hm[row] == 1)[0]
        if ones.size:
            ops = []
            for q in ones:
                ops += [data[int(q)], a]
            c.append("CX", ops)
    c.append("M", ancZ)

def append_X_stabilizers_golay(c: stim.Circuit, Hm: np.ndarray, data: list[int], ancX: list[int]):
    """Measure X-type stabilizers (detect Z component) using |+> ancillas and MX."""
    c.append("R", ancX)
    c.append("H", ancX)
    for row in range(12):
        a = ancX[row]
        ones = np.where(Hm[row] == 1)[0]
        if ones.size:
            ops = []
            for q in ones:
                ops += [a, data[int(q)]]
            c.append("CX", ops)
    c.append("MX", ancX)

def build_golay_S_bias_circuit(p_noise: float, apply_S: bool) -> stim.Circuit:
    Hm = build_golay24_H()
    data = list(range(0, 24))

    ancX0 = list(range(24, 36))  # prepX (12)
    ancZ  = list(range(36, 48))  # postZ (12)
    ancX1 = list(range(48, 60))  # postX (12)

    c = stim.Circuit()
    c.append("R", data + ancX0 + ancZ + ancX1)

    # Reference X-syndrome (baseline for X-check readout)
    append_X_stabilizers_golay(c, Hm, data, ancX0)  # 12 bits: prepX

    # X-only noise before the gate
    if p_noise > 0:
        c.append("X_ERROR", data, p_noise)

    # Optional transversal S
    if apply_S:
        c.append("S", data)

    # Post syndromes
    append_Z_stabilizers_golay(c, Hm, data, ancZ)   # 12 bits: postZ
    append_X_stabilizers_golay(c, Hm, data, ancX1)  # 12 bits: postX

    return c

def run_golay_S_bias_demo(p_noise: float, shots: int, seed: int = 23456, show_diagrams: bool = True):
    Hm = build_golay24_H()
    assert Hm.shape == (12, 24)

    def analyze(c: stim.Circuit, label: str):
        sampler = c.compile_sampler(seed=seed)
        ms = sampler.sample(shots).astype(np.uint8)

        # Layout: prepX(12) | postZ(12) | postX(12)
        prepX = ms[:, 0:12]
        postZ = ms[:, 12:24]
        postX = ms[:, 24:36]

        x_synd = postZ                 # baseline 0
        z_synd = postX ^ prepX         # baseline-subtracted

        trig_x = np.mean(np.any(x_synd != 0, axis=1)) * 100
        trig_z = np.mean(np.any(z_synd != 0, axis=1)) * 100

        # For X->Y under S, syndromes tend to match (same locations) shot-by-shot.
        match = np.mean(np.all(x_synd == z_synd, axis=1)) * 100

        print(f"\n=== {label} ===")
        print(f"p_noise = {p_noise:.4f} | shots = {shots}")
        print(f"X-syndrome triggers (detect X component): {trig_x:.2f}%")
        print(f"Z-syndrome triggers (detect Z component): {trig_z:.2f}%")
        print(f"Syndrome match rate (x_synd == z_synd):  {match:.2f}%")

    c0 = build_golay_S_bias_circuit(p_noise=p_noise, apply_S=False)
    cS = build_golay_S_bias_circuit(p_noise=p_noise, apply_S=True)

    if show_diagrams:
        print("\nCircuit: CONTROL (no S)")
        display(c0.diagram("timeline-svg"))
        print("\nCircuit: WITH transversal S")
        display(cS.diagram("timeline-svg"))

    analyze(c0, "CONTROL: no S")
    analyze(cS, "WITH S: transversal S (X -> Y)")

# --- RUN ---
p_noise = 0.05
shots = 20_000
run_golay_S_bias_demo(p_noise=p_noise, shots=shots, seed=23456, show_diagrams=True)

### The Phase Gate Anomaly at Scale: Why S-Gates Break Cat Qubits

While the Hadamard gate explicitly swaps $X$ and $Z$ axes, the Phase gate ($S$) presents a more subtle but equally devastating challenge for Cat Qubits. Because the $S$ gate only rotates around the Z-axis, it is tempting to assume it might preserve the bit-flip bias. To definitively test this, we applied our syndrome-differencing methodology to a 24-qubit Golay block under transversal $S$ execution.

**1. The Pure X-Noise Baseline (Control Scenario)**
We injected a 5% physical bit-flip ($X$) error rate into the 24-qubit data block.
* **Result Interpretation:** As established, the probability of at least one physical error occurring across 24 qubits at $p=0.05$ is ~71%. The "CONTROL" circuit confirms this exactly: the $X$-syndrome triggers fired at **71.34%**, while the $Z$-syndromes remained at **0.00%**. This confirms our tracking algorithm correctly isolates pure bit-flip noise without false positives. *(Note: The 28.66% match rate here simply represents the shots where exactly zero errors occurred, meaning both syndromes were naturally 0).*

**2. The Pauli Y Conjugation (With Transversal S Scenario)**
We then applied a direct transversal $S$ gate across the noisy data. The Pauli conjugation rules dictate that $S \cdot X \cdot S^\dagger = Y$. In the Pauli group, a $Y$ error is not a separate entity; it is the simultaneous occurrence of a bit-flip and a phase-flip on the same qubit ($Y = iXZ$).
* **Result Interpretation:** The output from the simulator perfectly captures this quantum mechanical transformation at scale. The $Z$-syndrome triggers spiked from 0% to **71.34%**, exactly matching the $X$-syndrome triggers. 

**3. The Ultimate Proof: 100.00% Syndrome Matching**
The most critical metric in this simulation is the **Syndrome match rate**, which hit exactly **100.00%**. 
This is the mathematical "smoking gun." It proves that the $Z$ errors didn't just appear randomly; they appeared in the *exact same locations* and in the *exact same patterns* as the original $X$ errors. Every single physical $X$ error in the 24-qubit block was simultaneously converted into an $X+Z$ error, triggering identical parity signatures across both the $X$ and $Z$ stabilizer checks.



**Conclusion**
This simulation unequivocally proves that applying a transversal $S$ gate to a Cat Qubit memory block instantaneously destroys the hardware bias by converting natural bit-flips into $Y$ errors, introducing massive, uncorrectable phase-flip components. 

*Architectural Verdict:* To build a viable fault-tolerant quantum computer with Cat Qubits and the Golay `[24,12,8]` code, **all phase-modifying gates (both $H$ and $S$) must be strictly executed via Gate Teleportation** using bias-preserving transversal CNOT routing.